In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
import ast
import folium
from branca.colormap import LinearColormap
from folium import Choropleth

In [ ]:
rent_data = pd.read_csv("../data/curated/rental_data.csv")

rent_data.head()

# Convert the string to a list of floats
rent_data['coordinates'] = rent_data['coordinates'].apply(ast.literal_eval)

# Convert the list of floats to Point geometry (note the order: (longitude, latitude))
rent_data['geometry'] = rent_data['coordinates'].apply(lambda x: Point(x[1], x[0]))

# Convert DataFrame to GeoDataFrame
rent_data_sf = gpd.GeoDataFrame(rent_data, geometry='geometry', crs="EPSG:7844")


In [ ]:
sa2_districts_sf = gpd.read_file("../data/landing/sa2/SA2_2021_AUST_GDA2020.shp")

In [ ]:
sa2_districts_sf.head(2)

In [ ]:
rent_data_sf = gpd.sjoin(rent_data_sf, sa2_districts_sf[['SA2_CODE21', 'SA2_NAME21', 'geometry']], how="left", op="within")
rent_data_sf.head(2)

In [ ]:
#can start by dropping all the 0 bedroom houses
df_filtered = rent_data_sf[rent_data_sf['beds'] != 0]

# need to deal with the low values and the high values
df_sorted = df_filtered.sort_values(by='Weekly Rent Amount')

#now drop the storage or car park values from the title
keywords_to_filter = ['storage', 'Storage', 'STORAGE', 'car']
mask = df_sorted['title'].str.contains('|'.join(keywords_to_filter), case=False)
df_filtered = df_sorted[~mask]


#dealing with the high rent values
#can drop all those > 10000 they are scuffed
df_filtered = df_filtered[df_filtered['Weekly Rent Amount'] < 10000]

#the per month ones are still showing dropping the remaining month ones 
df = df_filtered.reset_index(drop=True)

keywords_to_filter = ['Per Month', 'pm', 'PER MONTH', 'MONTH']
mask = df['cost_text'].str.contains('|'.join(keywords_to_filter), case=False)
df1 = df[mask]
df2 = df[~mask]

# Just drop the last 6 
names_to_drop = df1.tail(6)['name'].values
result_df = df1[~df1['name'].isin(names_to_drop)]

#joining the dataframes back together
joined_df = pd.concat([result_df, df2], axis=0)

# Reset the index of the result DataFrame if needed
joined_df.reset_index(drop=True, inplace=True)

#might want to deal with those bedroom for rent ones
df_sorted = joined_df.sort_values(by='Weekly Rent Amount')
df_sorted
rent_data_sf = df_sorted


In [ ]:
rent_data_sf = rent_data_sf[rent_data_sf['SA2_NAME21'] != 'Flemington Racecourse']

In [ ]:
#grouping by postcode
postcode_grouping = rent_data_sf.groupby(['Postcode', 'property type'])

In [ ]:
average_weekly_rent = postcode_grouping['Weekly Rent Amount'].mean()
average_beds = postcode_grouping['beds'].mean() # average cost per bedroom in each postcode
average_price_per_bed = average_weekly_rent /average_beds

house_ave_price_bed = average_price_per_bed[average_price_per_bed.index.get_level_values('property type') == 'House']

unit_ave_price_bed = average_price_per_bed[average_price_per_bed.index.get_level_values('property type') == 'Apartment / Unit / Flat']

unit_ave_price_bed.head()

#postcode_grouping['beds'].max().sort_values(ascending=False)
#postcode_grouping.count()

In [ ]:
#grouping by sa2 region
sa2_grouping = rent_data_sf.groupby(['SA2_NAME21', 'property type'])

result = rent_data_sf.groupby(['SA2_NAME21', 'property type']).agg({'Weekly Rent Amount': 'mean', 'geometry': 'first'})

In [ ]:
sa2_average_weekly_rent = sa2_grouping['Weekly Rent Amount'].mean()
sa2_average_beds = sa2_grouping['beds'].mean() # average cost per bedroom in each sa2 region
sa2_average_price_per_bed = sa2_average_weekly_rent /sa2_average_beds

sa2_house_ave_price_bed = sa2_average_price_per_bed[sa2_average_price_per_bed.index.get_level_values('property type') == 'House']

sa2_unit_ave_price_bed = sa2_average_price_per_bed[sa2_average_price_per_bed.index.get_level_values('property type') == 'Apartment / Unit / Flat']

sa2_unit_ave_price_bed.head()

In [ ]:
#I need to join the shapefile and the grouped dataframe
sa2_house_ave_price_bed = pd.DataFrame(sa2_house_ave_price_bed)
sa2_house_ave_price_bed.rename(columns={0: 'Average weekly rent per bed'}, inplace=True)
merged_df = sa2_house_ave_price_bed.merge(sa2_districts_sf, on=['SA2_NAME21', 'SA2_NAME21'], how='left')

#subsetting the columns I require
df = merged_df[['SA2_NAME21', 'Average weekly rent per bed', 'LOCI_URI21', 'geometry']]

# converting the dataframe into a geometry
df['Average weekly rent per bed'].replace(float('inf'), float('nan'), inplace=True)
gdf = gpd.GeoDataFrame(df, geometry='geometry')


In [ ]:
# Create a Folium map centered around Melbourne
m = folium.Map(location=[-37.8136, 144.9631], tiles="Stamen Terrain", zoom_start=7)

# Assuming you already have a GeoDataFrame 'gdf' with the geometry column and rent data
# You don't need to iterate through the GeoDataFrame, just use it directly

# Create a Choropleth layer
folium.Choropleth(
    geo_data=gdf['geometry'],
    data=gdf['Average weekly rent per bed'],
    columns=['geometry', 'Average weekly rent per bed'],  # Use the correct column names
    key_on='feature.id',  # Use 'feature.id' to match GeoJSON features
    fill_color='OrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Average Weekly Rent per Bed'
).add_to(m)

# Specify the folder and file path to save the map
file_path = '../plots/house_ave_price_bed_map.png'

m.save(file_path)

In [ ]:
m

In [ ]:
#I need to join the shapefile and the grouped dataframe
sa2_unit_ave_price_bed = pd.DataFrame(sa2_unit_ave_price_bed)
sa2_unit_ave_price_bed.rename(columns={0: 'Average weekly rent per bed'}, inplace=True)
merged_df = sa2_unit_ave_price_bed.merge(sa2_districts_sf, on=['SA2_NAME21', 'SA2_NAME21'], how='left')

#subsetting the columns I require
df = merged_df[['SA2_NAME21', 'Average weekly rent per bed', 'LOCI_URI21', 'geometry']]

# converting the dataframe into a geometry
df['Average weekly rent per bed'].replace(float('inf'), float('nan'), inplace=True)
gdf = gpd.GeoDataFrame(df, geometry='geometry')


In [ ]:
sa2_unit_ave_price_bed.sort_values('Average weekly rent per bed')

In [ ]:
# Create a Folium map centered around Melbourne
m = folium.Map(location=[-37.8136, 144.9631], tiles="Stamen Terrain", zoom_start=7)

# Assuming you already have a GeoDataFrame 'gdf' with the geometry column and rent data
# You don't need to iterate through the GeoDataFrame, just use it directly

# Create a Choropleth layer
folium.Choropleth(
    geo_data=gdf['geometry'],
    data=gdf['Average weekly rent per bed'],
    columns=['geometry', 'Average weekly rent per bed'],  # Use the correct column names
    key_on='feature.id',  # Use 'feature.id' to match GeoJSON features
    fill_color='OrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Average Weekly Rent per Bed'
).add_to(m)

file_path = '../plots/unit_ave_price_bed_map.png'

m.save(file_path)


In [ ]:
m

In [ ]:
#plot things like average number of beds, baths

#will also have to plot the crime stats.
